# 🕵️ Dia 15 — Vetores de ataque comuns e a evidência que cada um deixa

**Investigador Cyber 90 Dias** · Fase 1 · Rank Jutsu `D`

> Baseado no *Day 15* de [cyber-investigator-skills](https://github.com/thiefdroppers/cyber-investigator-skills/blob/main/curriculum/phase-1-foundations/day-15.md), © ThiefDroppers (MIT).
> Notebook, camada BR e Camada 8 por Deivison Lourenço ([@Ridd1kulusC0d3r](https://github.com/Ridd1kulusC0d3r)).

---

### 🎬 Onde paramos

Quarta, 7/out. O Dr. Hélio quer uma frase para o conselho: *"como eles entraram?"*. Não houve invasão do portal. Mas três cooperados tiveram transferências feitas por alguém que entrou **com a senha certa**.

Rafa: *"São dois vetores, não um. Faz o mapa de prontidão: vetor, técnica, registro, quem guarda, por quanto tempo."*

Neste notebook você vai:

1. **Pontuar** os vetores do original numa escala de 0 a 3, com justificativa.
2. **Gerar a camada do ATT&CK Navigator** em JSON, sem clicar em nada.
3. **Validar o JSON** antes de carregar (o teste do recarregamento, automatizado).
4. **Montar a camada da Serra Azul** e jogar o quiz "qual vetor, qual registro?".

🐍 Só biblioteca padrão do Python. Nada para instalar. Rode as células de cima para baixo (Shift+Enter). Tudo funciona **sem internet**; quando uma célula tenta a rede, ela tem plano B embutido.

## 1. As notas do cenário do original

Organização fictícia do Dia 9: e-mail/identidade em nuvem com registro padrão, UFW, logs web, `auth.log`, **sem** EDR nos notebooks. As notas abaixo são uma **proposta**; mude as que você discordar (o formulário aceita edição direta no dicionário).

In [ ]:
#@title Notas do cenário do original { display-mode: "form" }
# Formato: "ID": (nota 0-3, "fonte de log — limitação")
NOTAS_ORIGINAL = {
    "T1566":     (1, "Rastreamento de mensagens do e-mail em nuvem — retenção curta no plano básico; sem log do clique"),
    "T1566.001": (1, "Rastreamento de mensagens mostra o anexo — sem EDR, não se vê o que ele executou"),
    "T1566.002": (1, "Rastreamento mostra o link — sem log de DNS/proxy, não se vê quem clicou"),
    "T1078":     (2, "Logs de entrada do provedor de identidade — retenção de semanas no plano básico"),
    "T1110":     (2, "auth.log e logs de entrada — mostram falhas e sucesso; rotação de 4 semanas"),
    "T1110.003": (2, "Logs de entrada mostram uma senha em muitas contas — só se o provedor registrar falhas por conta"),
    "T1110.004": (1, "Stuffing de IPs rotativos — sem correlação central, cada log vê poucas tentativas"),
    "T1190":     (2, "Logs de acesso web capturam exploits na URL — não nos corpos de POST, sem log de processo"),
    "T1133":     (2, "UFW mostra a porta exposta e auth.log mostra o SSH — sem log de VPN informado"),
    "T1189":     (0, "Nenhuma fonte (sem proxy, DNS interno ou EDR) — nada mostra a visita ou o download"),
    "T1195":     (1, "Só avisos do fabricante e hashes publicados — nenhum log de atualização centralizado"),
    "T1199":     (1, "Contas do provedor aparecem nos logs de entrada — sem log da ferramenta de gerência remota"),
    "T1091":     (0, "Nenhuma fonte (sem EDR nem auditoria de USB) — conexão de pendrive não deixa registro central"),
    "T1200":     (0, "Nenhuma fonte de inventário de dispositivos — acesso físico não é registrado"),
    "T1684.001": (1, "Cabeçalhos e corpo do e-mail existem — chamadas e SMS ficam fora do alcance"),
}
print(len(NOTAS_ORIGINAL), "técnicas pontuadas (o checkpoint pede pelo menos 11).")

In [ ]:
# ------------------------------------------------------------
# Mapa de calor em texto: um bloco por nota, do pior para o melhor
# ------------------------------------------------------------
BARRA = {0: "🟥", 1: "🟧", 2: "🟨", 3: "🟩"}
LEGENDA = "0 = nenhuma fonte · 1 = parcial/curta/terceiro · 2 = provável, com lacunas · 3 = forte e retida"

def mapa(notas, titulo):
    print(f"{titulo}\n{LEGENDA}\n")
    for tid, (nota, just) in sorted(notas.items(), key=lambda x: x[1][0]):
        print(f"{BARRA[nota]} {nota}  {tid:<10} {just}")

mapa(NOTAS_ORIGINAL, "Prontidão de evidência — acesso inicial, organização fictícia (Dia 15)")

## 2. Gerar a camada do Navigator

O formato de camada é JSON e está documentado no [repositório do Navigator](https://github.com/mitre-attack/attack-navigator/tree/master/layers). Os campos que importam hoje: `techniqueID`, `score`, `comment`, e o `gradient` de 0 a 3.

In [ ]:
import json
from pathlib import Path

def gerar_camada(nome, descricao, notas):
    """Monta o dicionário da camada no formato do ATT&CK Navigator."""
    return {
        "name": nome,
        "versions": {"layer": "4.5", "navigator": "5.1.0"},   # versão do formato de camada (veja o repositório)
        "domain": "enterprise-attack",
        "description": descricao,
        "techniques": [
            {"techniqueID": tid, "score": nota, "comment": just, "enabled": True,
             "showSubtechniques": "." not in tid}             # expande a técnica-mãe para mostrar as subs
            for tid, (nota, just) in notas.items()
        ],
        "gradient": {"colors": ["#ff6666", "#ffe766", "#8ec843"], "minValue": 0, "maxValue": 3},
        "legendItems": [{"label": l, "color": c} for l, c in
                        [("0 nenhuma fonte", "#ff6666"), ("1 parcial", "#ffb366"),
                         ("2 provável, com lacunas", "#ffe766"), ("3 forte e retida", "#8ec843")]],
        "hideDisabled": False,
    }

camada_original = gerar_camada(
    "Initial access evidence readiness, fictional org (Day 15)",
    "Cenário do Dia 9 (VLANs, UFW, logs web e auth, e-mail em nuvem padrão, sem EDR). " + LEGENDA,
    NOTAS_ORIGINAL)

Path("notas").mkdir(exist_ok=True)
Path("notas/day15-initial-access.json").write_text(json.dumps(camada_original, ensure_ascii=False, indent=2), encoding="utf-8")
print("💾 notas/day15-initial-access.json — carregue no Navigator com 'Open Existing Layer'.")

In [ ]:
# ------------------------------------------------------------
# O "teste do recarregamento" (Passo 5), antes mesmo de abrir o Navigator:
# relê o arquivo do disco e confere se nada se perdeu.
# ------------------------------------------------------------
import re

def validar(caminho, notas):
    dado = json.loads(Path(caminho).read_text(encoding="utf-8"))          # relê do disco
    problemas = []
    ids = {t["techniqueID"]: t for t in dado["techniques"]}
    for tid, (nota, just) in notas.items():
        if not re.fullmatch(r"T\d{4}(\.\d{3})?", tid):
            problemas.append(f"{tid}: ID fora do formato Txxxx(.xxx)")
        t = ids.get(tid)
        if t is None:
            problemas.append(f"{tid}: sumiu do arquivo")
        elif t["score"] != nota or t["comment"] != just:
            problemas.append(f"{tid}: nota ou comentário diferente")
        if " — " not in just:
            problemas.append(f"{tid}: justificativa sem 'fonte — limitação'")
    g = dado["gradient"]
    if (g["minValue"], g["maxValue"]) != (0, 3):
        problemas.append("gradiente não vai de 0 a 3")
    print(f"{caminho}: {len(ids)} técnicas;", "✅ tudo intacto" if not problemas else "❌")
    for p in problemas:
        print("   -", p)
    return not problemas

validar("notas/day15-initial-access.json", NOTAS_ORIGINAL)

## 3. ➕ A camada da Serra Azul

Os vetores possíveis do caso, com o que a cooperativa tem (README, Passo 6). Ajuste as notas e as justificativas.

In [ ]:
#@title Notas da Operação Isca { display-mode: "form" }
NOTAS_ISCA = {
    "T1566.002": (1, "Prints de SMS da ouvidoria e hotlinks do Dia 11 — o SMS chega ao celular pessoal, nada no gateway da cooperativa"),
    "T1684.001": (2, "Texto do SMS, remetentes +55 00 00000-0001 a -0004 e encurtador lnk.example/sa7 nos prints — sem registro da operadora"),
    "T1078":     (1, "Logs de entrada do internet banking — com a central, 90 dias, entrega por chamado em 5 dias úteis"),
    "T1110":     (1, "Mesmos logs do internet banking descartariam força bruta (falhas antes do sucesso) — ainda não recebidos"),
    "T1190":     (2, "access.log e auth.log do portal (Dias 11, 12, 14) — sem log de aplicação nem de processo"),
}
recomendacao = "Abrir hoje chamado à central pedindo PRESERVAÇÃO dos logs de entrada do internet banking das 3 contas afetadas em 5/out (retenção de 90 dias: limite 3/jan/2027)."  #@param {type:"string"}

camada_isca = gerar_camada("Serra Azul — prontidão de evidência, Operação Isca (Dia 15)",
                           "Vetores possíveis do golpe do 'Pix bloqueado'. " + LEGENDA, NOTAS_ISCA)
Path("notas/isca-dia15-serraazul.json").write_text(json.dumps(camada_isca, ensure_ascii=False, indent=2), encoding="utf-8")
validar("notas/isca-dia15-serraazul.json", NOTAS_ISCA)
print()
mapa(NOTAS_ISCA, "Serra Azul — Operação Isca")
print("\nRecomendação:", recomendacao)

> 🧠 Repare que nenhuma nota passa de 2. O "vetor" que mais pesa no caso é a decisão de uma pessoa assustada diante de um SMS, e esse registro está na memória e no celular dela, não num log. Por isso os prints da ouvidoria são evidência: com data, hora e sem culpar quem clicou.

## 4. 🎮 Quiz: qual vetor, qual registro?

In [ ]:
MODO_JOGO = True

QUIZ = [
    ("Cinco falhas e um sucesso do mesmo IP em 2 s", ["a) T1110", "b) T1189", "c) T1195"], "a", "Força bruta/palpite que funcionou (Dia 11 e 12)."),
    ("Login de um país novo com a senha certa, sem falhas antes", ["a) T1190", "b) T1078", "c) T1091"], "b", "Credencial válida; pergunte como foi obtida."),
    ("Onde ver se um e-mail passou no SPF/DKIM/DMARC?", ["a) No cabeçalho Authentication-Results", "b) No auth.log", "c) No firewall"], "a", "O próprio cabeçalho da mensagem."),
    ("RDP com logon tipo 10 vindo da internet", ["a) T1133", "b) T1566", "c) T1199"], "a", "Serviço remoto externo; evento 4624 tipo 10."),
    ("Na Serra Azul, como o SMS falso é classificado?", ["a) T1190", "b) T1566.002 e T1684.001", "c) T1200"], "b", "Link na mensagem + personificação da cooperativa."),
    ("E o uso da senha no internet banking?", ["a) T1078", "b) T1110.004", "c) T1195"], "a", "Credencial válida entregue pela vítima."),
    ("A evidência de T1078 no caso está...", ["a) no access.log do portal", "b) nos logs do internet banking, com a central, 90 dias", "c) no celular da vítima"], "b", "Terceiro + retenção curta = pedir preservação já."),
    ("A diretoria pergunta se 'o sistema foi invadido'. Resposta correta:", ["a) Sim", "b) Não houve invasão; houve personificação e uso de credencial válida", "c) Não sabemos e não dá para saber"], "b", "Dois vetores, nenhum é exploração do sistema."),
]
acertos = 0
for i, (p, ops, certa, porque) in enumerate(QUIZ, start=1):
    print(f"\n[{i}/{len(QUIZ)}] {p}")
    for o in ops:
        print("   ", o)
    if MODO_JOGO:
        r = ""
        while r not in ("a", "b", "c"):
            r = input("   Resposta (a/b/c): ").strip().lower()
        acertos += (r == certa)
        print("   ✅" if r == certa else f"   ❌ Era {certa}.")
    print("   Por quê:", porque)
if MODO_JOGO:
    print(f"\n🏁 {acertos}/{len(QUIZ)}", "— Checkpoint BR cumprido." if acertos >= 6 else "— Meta: 6.")

## 5. 📥 Baixar as camadas

In [ ]:
md = "# Notas de prontidão — Serra Azul (Dia 15)\n\n" + LEGENDA + "\n\n| Técnica | Nota | Fonte — limitação |\n|---|---|---|\n"
md += "".join(f"| {t} | {n} | {j} |\n" for t, (n, j) in NOTAS_ISCA.items())
md += "\n## Recomendação\n" + recomendacao + "\n"
Path("notas/isca-dia15-scores.md").write_text(md, encoding="utf-8")
print(md)
try:
    from google.colab import files
    for f in ("day15-initial-access.json", "isca-dia15-serraazul.json", "isca-dia15-scores.md"):
        files.download(f"notas/{f}")
except ImportError:
    print("(Fora do Colab: os arquivos ficaram em notas/)")

## ✅ Antes de fechar o Dia 15

- [ ] Quiz com pelo menos 6/8
- [ ] As duas camadas carregadas no Navigator com 'Open Existing Layer' e exportadas em SVG
- [ ] `day15-scores.md` e `isca-dia15-scores.md` com fonte e limitação em cada linha
- [ ] `status` do dia 15 = `estudado` no `curriculo/mapa-dos-dias.csv`

**Amanhã (Dia 16):** modelagem de ameaças com STRIDE. A Marta quer saber o que **mais** pode dar errado, e o primeiro alvo é o próprio canal de atendimento da cooperativa.

---
<sub>Baseado em cyber-investigator-skills por ThiefDroppers (MIT) · Investigador Cyber 90 Dias por Deivison Lourenço</sub>